# Open X-Embodiment × FiftyOne — Physical AI Demo

Explore an **Open X-Embodiment (OXE)** robotics dataset in **FiftyOne 1.19**, loaded via the
community **LeRobot v3.0 importer**. OXE pools 1M+ real robot trajectories from 60 datasets
across 34 labs and 22 embodiments — a heterogeneous corpus that's a compelling data-curation story.

Most OXE datasets on the Hub (the `IPEC-COMMUNITY` conversions) are stored in **LeRobot
v2.1**, but the FiftyOne importer needs **v3.0**. So the real pipeline is:

1. **Validate** a candidate dataset on the Hub (declared cameras vs. actual video) — cheap, metadata-only
2. **Download** it fully, then **verify the local copy** is complete and balanced
3. **Convert** v2.1 → v3.0 locally (no Hub push)
4. **Import** into FiftyOne as a grouped video dataset (one group per episode, cameras as slices)
5. **Explore**: episodes, tasks, multi-camera views, embeddings, similarity search

We use **`berkeley_fanuc_manipulation`**: a manipulation dataset with **2 cameras**
(`image` + `wrist_image`), **415 episodes**, **32 tasks** — small enough to run fast, rich
enough for a real embeddings plot.

> **Pinned to FiftyOne 1.19** on purpose — 1.20 has an embedding-panel bug, and the
> embeddings step relies on that panel.

**Reproducible on any machine.** Complete the terminal setup in cell 0, then run the cells
top to bottom. To try a **different** OXE dataset, change `REPO_ID` in cell 3 (validate it
first with cell 2) and update the converted-path in cell 5 to match its repo name — the rest
of the notebook is dataset-agnostic (camera slices, tasks, and action dims are all discovered
at runtime, not hardcoded). Note that the **action layout in cell 7b** (`action[:6]` = arm,
`action[6]` = gripper) is specific to this robot; other embodiments differ, so check the field
description printed in cell 7b and adjust the indices if you switch datasets.


## 0. Environment setup (run these in a terminal, once)

Do **not** run this inside the notebook — set up a clean, isolated virtual environment
first, then launch Jupyter from inside it so this notebook runs against the right kernel.

FiftyOne 1.19 supports **Python 3.10–3.12**. Newer Pythons (3.13/3.14) don't yet have
reliable wheels for the ML stack, so target **3.11**. Steps below are cross-platform; where a
command differs by OS, both are shown.

```bash
# 1. Get Python 3.11 if you don't have it
#    macOS (Homebrew):     brew install python@3.11
#    Ubuntu/Debian:        sudo apt install python3.11 python3.11-venv
#    Windows:              install from python.org, or `winget install Python.Python.3.11`
#    conda (any OS):       conda create -n oxe-demo python=3.11 && conda activate oxe-demo

# 2. Create + activate a dedicated venv (skip if you used conda above)
python3.11 -m venv ~/oxe-demo                 # Windows: py -3.11 -m venv %USERPROFILE%\oxe-demo
source ~/oxe-demo/bin/activate                # Windows: %USERPROFILE%\oxe-demo\Scripts\activate
python --version                               # -> Python 3.11.x

# 3. Upgrade pip
pip install --upgrade pip

# 4. Pin FiftyOne to 1.19 (avoids the 1.20 embedding-panel bug)
pip install "fiftyone==1.19.*"

# 5. Importer/Hub deps, embeddings deps, and LeRobot (for the v2.1->v3.0 converter)
pip install ffmpeg-python pyarrow huggingface_hub umap-learn torch torchvision lerobot

# 6. Jupyter kernel inside the venv
pip install jupyter ipykernel
python -m ipykernel install --user --name oxe-demo --display-name "Python (oxe-demo)"

# 7. ffmpeg with AV1 support (clips are AV1; the importer re-encodes to H.264)
#    macOS:            brew install ffmpeg
#    Ubuntu/Debian:    sudo apt install ffmpeg
#    conda (any OS):   conda install -c conda-forge ffmpeg
#    Windows:          winget install Gyan.FFmpeg   (or download from ffmpeg.org)
ffmpeg -decoders | grep av1                    # expect libdav1d / libaom-av1 / av1

# 8. Authenticate with Hugging Face (the CLI is `hf`, not the old `huggingface-cli`).
#    A free account + read token is enough; logging in avoids strict anonymous rate limits.
hf auth login                                  # paste a read token from hf.co/settings/tokens
hf auth whoami

# 9. Clone the importer INTO THE SAME FOLDER as this notebook (cell 1 uses a relative path)
cd /path/to/folder/with/this/notebook
git clone https://github.com/harpreetsahota204/fiftyone_lerobot_importer.git

# 10. Disable HF Xet transfer (avoids intermittent 429s during download AND conversion).
#     Set it in the SAME shell you launch Jupyter and run the converter from.
#     macOS/Linux (bash/zsh):   export HF_HUB_DISABLE_XET=1
#     to persist:               echo 'export HF_HUB_DISABLE_XET=1' >> ~/.bashrc   (or ~/.zshrc)
#     Windows (PowerShell):     $env:HF_HUB_DISABLE_XET = "1"
export HF_HUB_DISABLE_XET=1

# 11. Launch Jupyter from inside the venv
jupyter notebook
# In Jupyter: open this notebook, then Kernel -> Change Kernel -> "Python (oxe-demo)"
```

**Hardware note:** the full run imports 415 episodes and builds a ~62k-frame CLIP index
(~20 min on CPU). A GPU speeds up embeddings substantially. For a quick first pass, set
`max_samples` small in cell 6 (see the note there) — everything works at any size.


## 1. Verify the environment

Confirm the kernel is the venv, FiftyOne is 1.19, and the importer is importable.
`HF_HUB_DISABLE_XET` is set here too, defensively — but the **converter** (step 5) runs in a
*separate terminal process*, so it needs the var in that shell as well (the `~/.zshrc` line in
cell 0 handles that).

In [ ]:
import sys, os

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

_candidates = [
    os.path.abspath("fiftyone_lerobot_importer"),
    os.path.expanduser("~/fiftyone_lerobot_importer"),
]
IMPORTER_DIR = next((p for p in _candidates
                     if os.path.isfile(os.path.join(p, "lerobot_importer.py"))), None)
if IMPORTER_DIR is None:
    raise FileNotFoundError(
        "Could not find lerobot_importer.py in:\n  " + "\n  ".join(_candidates)
        + "\n\nClone it into this notebook's folder and re-run:\n"
        + "    git clone https://github.com/harpreetsahota204/fiftyone_lerobot_importer.git\n"
        + f"(This notebook's working dir is: {os.getcwd()})"
    )
if IMPORTER_DIR not in sys.path:
    sys.path.insert(0, IMPORTER_DIR)

import fiftyone as fo
import fiftyone.brain as fob
from fiftyone import ViewField as F

print("Python     :", sys.version.split()[0])
print("FiftyOne   :", fo.__version__)
assert fo.__version__.startswith("1.19"), "Pin FiftyOne to 1.19 — see setup cell 0."

from lerobot_importer import LeRobotDataset, apply_lerobot_field_descriptions
print("Importer   : loaded from", IMPORTER_DIR)
print("HF_HUB_DISABLE_XET:", os.environ.get("HF_HUB_DISABLE_XET"))

## 2. Validate a dataset on the Hub *before* downloading

Some IPEC-COMMUNITY conversions declare a camera in their schema that has no video files.
Converting those fails late with `All cams dont have same number of episodes ([N, 0])`. This
check pulls only `meta/info.json` + the repo file listing (no bulk data) and cross-references
declared cameras against the actual `.mp4` count per camera on the Hub.

**Caveat:** validates the *remote repo* only — not whether your local download finished
(that's cell 4).

In [ ]:
import json
from collections import defaultdict
from huggingface_hub import HfApi, hf_hub_download

def validate_hub_dataset(repo_id):
    api = HfApi()
    print(f"\n=== {repo_id} ===")
    info = json.load(open(hf_hub_download(repo_id, repo_type="dataset", filename="meta/info.json")))
    version = info.get("codebase_version", "?")
    declared = [k for k in info.get("features", {}) if k.startswith("observation.images.")]
    print(f"  codebase_version : {version}")
    print(f"  total_episodes   : {info.get('total_episodes', '?')}")
    print(f"  declared cameras : {declared or '(none)'}")

    counts = defaultdict(int)
    for f in api.list_repo_files(repo_id, repo_type="dataset"):
        if f.startswith("videos/") and f.endswith(".mp4"):
            for cam in declared:
                short = cam.split("observation.images.")[-1]
                if f"/{cam}/" in f or f"/{short}/" in f:
                    counts[cam] += 1
                    break

    vals = [counts.get(c, 0) for c in declared]
    for cam in declared:
        n = counts.get(cam, 0)
        print(f"      {cam:<45} {n:>6}  {'OK' if n else '!! NO VIDEOS'}")
    safe = bool(declared) and all(vals) and len(set(vals)) == 1
    print("  VERDICT:", "✅ balanced — safe to convert" if safe
          else "⚠️ unbalanced/missing — converter will fail")
    return {"repo_id": repo_id, "version": version, "safe": safe, "counts": dict(counts)}

for repo in [
    "IPEC-COMMUNITY/berkeley_fanuc_manipulation_lerobot",  # 2 cams, 415 eps — our pick
    # "IPEC-COMMUNITY/taco_play_lerobot",                  # 2 cams, 3242 eps
    # "IPEC-COMMUNITY/bridge_orig_lerobot",                # 4 cams, 53k eps (large)
]:
    validate_hub_dataset(repo)

## 3. Download the dataset (retry + Xet disabled)

`snapshot_download` **resumes** — already-fetched files are skipped — so if it fails partway,
re-run. The retry loop rides out transient `429 Too Many Requests`. Fanuc is ~330 MB, quick.

In [ ]:
import time
from huggingface_hub import snapshot_download
from huggingface_hub.utils import HfHubHTTPError

REPO_ID   = "IPEC-COMMUNITY/berkeley_fanuc_manipulation_lerobot"
LOCAL_DIR = "oxe_fanuc"

def download_with_retry(repo_id, local_dir, max_retries=5):
    for attempt in range(1, max_retries + 1):
        try:
            return snapshot_download(repo_id=repo_id, repo_type="dataset",
                                     local_dir=local_dir, max_workers=4)
        except (HfHubHTTPError, RuntimeError, ConnectionError) as e:
            if "429" in str(e) and attempt < max_retries:
                wait = 10 * attempt
                print(f"[attempt {attempt}] rate-limited (429); retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("Exhausted retries")

download_path = download_with_retry(REPO_ID, LOCAL_DIR)
print("Downloaded to:", download_path)

## 4. Verify the LOCAL copy is complete and balanced

Cell 2 checks the remote repo; a rate-limited download can still leave one camera with 0 files
locally. This walks the downloaded `videos/` tree and counts `.mp4` per camera, matching camera
names as **whole path segments** (so `image` isn't confused with `wrist_image`).

In [ ]:
import os, json
from collections import defaultdict

def verify_local_download(local_dir):
    info = json.load(open(os.path.join(local_dir, "meta", "info.json")))
    cams = [k for k in info.get("features", {}) if k.startswith("observation.images.")]
    counts = defaultdict(int)
    for root, _, files in os.walk(os.path.join(local_dir, "videos")):
        segments = set(root.split(os.sep))     # exact segments — no substring bleed
        for f in files:
            if not f.endswith(".mp4"):
                continue
            for cam in cams:
                short = cam.split("observation.images.")[-1]
                if cam in segments or short in segments:
                    counts[cam] += 1
                    break
    print("Cameras declared :", cams)
    print("Local video/cam  :", dict(counts))
    vals = [counts.get(c, 0) for c in cams]
    ok = bool(vals) and all(vals) and len(set(vals)) == 1
    print("✅ complete & balanced" if ok else "❌ incomplete — re-run the download cell")
    return ok

assert verify_local_download(LOCAL_DIR), "Fix the download before converting."

## 5. Convert v2.1 → v3.0 (run in a terminal)

The importer needs LeRobot **v3.0**; IPEC-COMMUNITY datasets are **v2.1**. Two flags matter:

- **`--push-to-hub 0`** — do NOT upload. Without it the converter tries to commit back to the
  `IPEC-COMMUNITY` repo and fails `403 Forbidden` (you have read, not write). The real work
  happens locally *before* that push, but the 403 aborts and cleans up the output — so disable it.
- **`--root ~/oxe_convert`** — write output to a directory you own.

Run in a **terminal** (same shell, `HF_HUB_DISABLE_XET=1` from cell 0's `~/.zshrc`):

```bash
export HF_HUB_DISABLE_XET=1
python -m lerobot.datasets.v30.convert_dataset_v21_to_v30 \
    --repo-id=IPEC-COMMUNITY/berkeley_fanuc_manipulation_lerobot \
    --push-to-hub 0 \
    --root ~/oxe_convert \
    --force-conversion
```

**Where the output lands:** the converter does an in-place upgrade + rename. The **v3.0
result** is the folder with **no suffix**; your original **v2.1** becomes `..._old`:

```
~/oxe_convert/IPEC-COMMUNITY/berkeley_fanuc_manipulation_lerobot        <- v3.0 (use this)
~/oxe_convert/IPEC-COMMUNITY/berkeley_fanuc_manipulation_lerobot_old    <- v2.1 (original)
```

v3.0 **consolidates** episodes: all 415 frames → one `data/chunk-000/file-000.parquet`, and
each camera's 415 clips → one `videos/<cam>/chunk-000/file-000.mp4`. A couple of shard files
is correct — that's the format, not a truncated conversion.

In [ ]:
V30 = os.path.expanduser("~/oxe_convert/IPEC-COMMUNITY/berkeley_fanuc_manipulation_lerobot")

info = json.load(open(os.path.join(V30, "meta", "info.json")))
print("codebase_version:", info.get("codebase_version"))   # expect v3.0
print("cameras:", [k for k in info.get("features", {}) if k.startswith("observation.images.")])

for sub in ("videos", "data"):
    hits = [os.path.join(r, f)
            for r, _, fs in os.walk(os.path.join(V30, sub))
            for f in fs if f.endswith((".mp4", ".parquet"))]
    print(f"{sub}: {len(hits)} shard(s)")
assert info.get("codebase_version", "").startswith("v3"), "Not v3.0 — re-run the converter."

## 6. Import into FiftyOne as a grouped video dataset

One group per episode, each camera a group slice; frame fields parsed dynamically; AV1 clips
re-encoded to H.264 for App playback. `max_samples` caps the first import.

In [ ]:
DATASET_NAME = "oxe_fanuc_demo"

dataset = fo.Dataset.from_dir(
    dataset_dir=V30,
    dataset_type=LeRobotDataset,
    name=DATASET_NAME,
    max_samples=None,          # full 415 episodes for the demo build; set 25 for fast iteration
    include_frame_data=True,   # per-frame observation_state / action / timestamp
    overwrite=True,            # clean rebuild every run (avoids "name not available")
)
dataset.persistent = True      # NOTE: the attribute is `persistent`, not `persist`
apply_lerobot_field_descriptions(dataset)

print(dataset)
print("\nGroup slices (cameras):", dataset.group_slices)   # -> ['image', 'wrist_image']

### Launch the App

Use the **group slices dropdown** (top-left) to flip between `image` and `wrist_image` for the
same episode; scrub the timeline to watch the trajectory.

In [ ]:
session = fo.launch_app(dataset)

## 7. Explore: tasks and episodes

32 tasks, each a natural-language instruction — the first thing a curator slices by.

In [ ]:
print("Task counts:")
for task, n in dataset.count_values("task").items():
    print(f"  {n:>4}  {task}")

tasks = dataset.distinct("task")
if tasks:
    task_view = dataset.match(F("task") == tasks[0])
    print(f"\n{task_view.count()} samples for task: {tasks[0]!r}")
    session.view = task_view

In [ ]:
# Quick look at the episode-grouped view (the saved version is baked in cell 7c)
episodes_view = dataset.group_by("episode_index", order_by="frame_index")
print(episodes_view)

## 7b. Derive per-frame fields — make trajectories queryable

The raw frame fields `observation_state` and `action` are float arrays: powerful, but opaque
in the App (you can't histogram or slide-filter a list). Here we pull scalar quantities out of
them so they become sidebar-filterable — turning "arrays buried in a parquet file" into numbers
you can *see the distribution of and drag to filter*.

For `berkeley_fanuc` the 7-dim `action` is a **6-DOF end-effector delta + gripper**, so we
split it into `arm_action_magnitude` (how much the arm is moving) and `gripper` (0..1 grip
state) — two independent signals that drive two distinct demo beats below. Always confirm the
layout via the printed field description; other embodiments differ.

In [ ]:
import numpy as np

# Inspect what the action/state dimensions mean for this dataset.
# For berkeley_fanuc the 7-dim action is a 6-DOF end-effector delta + gripper:
#   action = [dx, dy, dz, droll, dpitch, dyaw, gripper]
for fname in ("frames.action", "frames.observation_state"):
    fld = dataset.get_field(fname)
    print(fname, "->", getattr(fld, "description", None))

# Derive scalar per-frame fields so they're sidebar-filterable
for sample in dataset.iter_samples(autosave=True, progress=True):
    for frame in sample.frames.values():
        if frame.action:
            a = np.asarray(frame.action, dtype=float)
            frame["action_magnitude"] = float(np.linalg.norm(a))          # full vector (reference)
            frame["arm_action_magnitude"] = float(np.linalg.norm(a[:6]))  # 6-DOF pose delta only
            frame["gripper"] = float(a[6])                                # dim 6 = gripper (0..1)
        if frame.observation_state:
            s = np.asarray(frame.observation_state, dtype=float)
            frame["state_norm"] = float(np.linalg.norm(s))

dataset.add_dynamic_frame_fields()   # register new frame fields on the schema
print("\nDerived: action_magnitude, arm_action_magnitude, gripper, state_norm")

## 7c. Pre-bake demo views (skip live code during the demo)

Everything below saves **named views** into the dataset. In the App, the saved-views dropdown
(top-left, next to the view bar) lets you jump between them with a single click — no code, no
typing, nothing to fail live. Run this once before the demo; then drive entirely from the App.

The views correspond to the demo beats:
- **by_episodes** — episodes grouped, frames ordered (navigation)
- **task_&lt;n&gt;** — one view per task, for the task-filtering beat
- **gripper_active** — frames where the gripper is engaged (grasp/release moments)
- **pure_transit** — arm moving while the gripper is idle (transit moments the gripper view excludes)

The last two are **orthogonal by construction**, so switching between them visibly changes the
grid — the clearest way to show that per-frame trajectory data is queryable, not opaque.

> Embeddings/similarity aren't saved as views here — they're driven from the Embeddings panel
> against the frames view (cells 8-9). Load those brain keys live in the panel.

In [ ]:
from fiftyone import ViewField as F

# 1. Episode navigation view (grouped, frames ordered)
dataset.save_view("by_episodes",
                  dataset.group_by("episode_index", order_by="frame_index"),
                  overwrite=True)

# 2. One saved view per task (dropdown becomes your task menu)
tasks = dataset.distinct("task")
for i, t in enumerate(tasks):
    dataset.save_view(f"task_{i:02d}", dataset.match(F("task") == t), overwrite=True)
print(f"Saved {len(tasks)} per-task views (task_00 ... task_{len(tasks)-1:02d})")

# 3. Per-frame filtering payoffs — the "trajectories are queryable" moment.
#    match_frames requires a VIDEO collection, so select one camera slice first.
cam = dataset.group_slices[0]                     # e.g. "image"
vv = dataset.select_group_slices(cam)             # media_type -> video

# Two ORTHOGONAL beats (thresholds locked to the full-415 distribution):
#   gripper_active : gripper engaged            (grasp / release moments)
#   pure_transit   : arm moving, gripper idle   (transit moments the gripper view excludes)
dataset.save_view("gripper_active",
                  vv.match_frames(F("gripper").abs() > 0.5),
                  overwrite=True)
dataset.save_view("pure_transit",
                  vv.match_frames((F("arm_action_magnitude") > 0.02) & (F("gripper").abs() < 0.5)),
                  overwrite=True)

# 4. Flattened FRAMES view for the histogram + slider beat.
#    to_frames() must be told to carry the derived per-frame fields via include_fields,
#    otherwise the sidebar won't show gripper / arm_action_magnitude (they'd be dropped).
frames_view = vv.to_frames(
    sample_frames=True,
    include_fields=["gripper", "arm_action_magnitude", "state_norm",
                    "observation_state", "action", "timestamp"],
)
dataset.save_view("frames_explore", frames_view, overwrite=True)

# to_frames() does NOT carry sample-level fields (like `task`) onto frames, and we must
# NOT re-flatten to add them (that mints new frame IDs and orphans the embeddings/similarity
# indexes). Instead, propagate `task` IN PLACE via each frame's parent sample_id — this
# preserves frame IDs so oxe_embeddings / oxe_sim stay valid, and enables Color-by task.
_id_to_task = dict(zip(vv.values("id"), vv.values("task")))
for _fr in frames_view.iter_samples(autosave=True, progress=True):
    _fr["task"] = _id_to_task.get(_fr.sample_id)
frames_view.reload()
print("frames_explore:", frames_view.count(), "frames | task present:",
      "task" in frames_view.get_field_schema())

print("Saved views:", dataset.list_saved_views())

### During the demo

1. `session = fo.launch_app(dataset)` (already running from cell 6).
2. Use the **saved-views dropdown** to click through: `by_episodes` -> a `task_*` view ->
   `gripper_active` -> `pure_transit`.
3. For the multi-camera beat, use the **group-slice dropdown** to flip `image` <-> `wrist_image`.
4. For the histogram + slider beat, load **`frames_explore`**, click `gripper` in the sidebar,
   and drag the range slider — the grid culls to grasp frames live.
5. For embeddings, open the **Embeddings panel** and pick the `oxe_embeddings` brain key.
   For similarity, run the `sort_by_similarity` cell (cell 9) — it pushes results to the App
   view. (The App's Similarity Search *panel* can't resolve a frames-view index in 1.19, so
   drive it from code and keep that panel closed.)

That's the demo driven mostly by clicks, with one code cell for semantic search.

## 8. Embeddings — reveal structure across the data

CLIP embeddings on one camera view, visualized in 2D via the FiftyOne Brain. Clusters
correspond to scenes, tasks, and trajectory phases. (This panel is broken in 1.20 — hence the
1.19 pin.)

**Why frames, not videos:** the samples here are *video clips* of varying length. Embedding a
video yields a per-frame array whose shape depends on the clip's frame count, so stacking
across clips fails (`all input arrays must have the same shape`). We convert to a **frames
view** first — one image sample per frame — so each embeds to a single fixed-length vector.
This also makes a richer plot: points are frames, so trajectory phases show up as structure.

`to_frames(sample_frames=True)` extracts frames to disk on first run, so it takes a moment.

In [ ]:
import warnings
# sklearn's cosine matmul emits benign RuntimeWarnings on large indexes; silence for clean output
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn")

# Reuse the frames view baked in cell 7c (same one used for the histogram beat)
frames_view = dataset.load_saved_view("frames_explore")
print(f"{frames_view.count()} frames to embed")

if "oxe_embeddings" in dataset.list_brain_runs():
    dataset.delete_brain_run("oxe_embeddings")

results = fob.compute_visualization(
    frames_view,
    model="clip-vit-base32-torch",           # downloaded from the Model Zoo on first run
    method="umap",
    brain_key="oxe_embeddings",
    batch_size=16,
)
print("Done. Open the Embeddings panel in the App and select 'oxe_embeddings'.")

In [ ]:
session.load_saved_view("frames_explore")
# In the App: + -> Embeddings panel -> brain key "oxe_embeddings".
# Color by "task" (propagated onto frames in cell 7c) to see scenes cluster by task.

## 8b. Find visual outliers — `compute_uniqueness`

Even one sub-dataset has structure worth auditing. `compute_uniqueness` scores every frame by
how visually distinct it is from the rest (reusing embeddings, so it's fast). Sort descending
to surface anomalies — odd lighting, occlusions, a dropped object, camera glitches — the frames
a curator wants to eyeball. We freeze the top 25 into a clickable saved view.

In [ ]:
frames_view = dataset.load_saved_view("frames_explore")

if "uniqueness" not in frames_view.get_field_schema():
    fob.compute_uniqueness(frames_view)   # adds a per-sample "uniqueness" float in [0, 1]

# Freeze the most-unique frames into a one-click saved view
outlier_ids = (frames_view
               .sort_by("uniqueness", reverse=True)
               .limit(25)
               .values("id"))
dataset.save_view(
    "visual_outliers",
    dataset.load_saved_view("frames_explore").select(outlier_ids, ordered=True),
    overwrite=True,
)
print("saved 'visual_outliers' (top 25 by uniqueness)")

# In the App you can also open the `uniqueness` field in the sidebar (it's a float, so you
# get a histogram + slider) and drag toward 1.0 to reveal the outliers live.

## 8c. Spot near-duplicate frames — `find_duplicates`

Robot video is full of near-duplicates: consecutive frames where nothing moves. The similarity
index can flag them via `find_duplicates(thresh=...)` (lower thresh = stricter). We collect the
flagged frames into a capped, clickable saved view. For training you'd often dedupe these so a
model doesn't over-weight static moments.

In [ ]:
sim = dataset.load_brain_results("oxe_sim")
sim.find_duplicates(thresh=0.04)     # tune: lower = stricter near-duplicate matching

# neighbors_map shape varies by version: values may be flat id lists or (id, dist) tuples.
# Handle both, and cap the number of duplicate GROUPS so the view stays browseable.
MAX_GROUPS = 100
dup_ids = set()
for i, (rep_id, neighbors) in enumerate(sim.neighbors_map.items()):
    if i >= MAX_GROUPS:
        break
    dup_ids.add(rep_id)
    for n in neighbors:
        dup_ids.add(n[0] if isinstance(n, (tuple, list)) else n)

dup_ids = list(dup_ids)
print(f"duplicate neighbor sets: {len(sim.neighbors_map)} | "
      f"frames in first {MAX_GROUPS} groups: {len(dup_ids)}")

dataset.save_view(
    "duplicate_frames",
    dataset.load_saved_view("frames_explore").select(dup_ids),
    overwrite=True,
)
print("saved 'duplicate_frames'")

## 9. Similarity search — find frames like this one

Index the **`frames_explore`** view (flat images — required, since embedding a *video*
collection produces ragged per-frame arrays that can't be stacked). CLIP is text-capable, so
this enables **natural-language image search** ("laptop", "red cup").

**Important — drive similarity from code, not the App's Similarity Search panel.** In
FiftyOne 1.19 the in-App panel can't resolve an index built on a `to_frames()` generated view
(its scopes key to base-dataset IDs, not frame-view IDs), so the panel throws
`Found array with 0 sample(s) (shape=(0, 512))`. Running `sort_by_similarity()` in a cell and
pushing the result to `session.view` works reliably and demos the same capability. Keep the
App's Similarity Search panel closed during the demo.

At ~62k frames this index takes ~20 min to build (one-time prep). First query is slower;
subsequent queries are fast.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="sklearn")

frames_view = dataset.load_saved_view("frames_explore")

# A brain key can exist but have empty/corrupt results (e.g. a prior build crashed
# mid-run). Check that results actually LOAD, not just that the key is registered.
def _sim_ready(ds, key):
    if key not in ds.list_brain_runs():
        return False
    try:
        return ds.load_brain_results(key) is not None
    except Exception:
        return False

if not _sim_ready(dataset, "oxe_sim"):
    if "oxe_sim" in dataset.list_brain_runs():
        dataset.delete_brain_run("oxe_sim")     # clear any dangling/empty run
    fob.compute_similarity(
        frames_view,
        model="clip-vit-base32-torch",
        brain_key="oxe_sim",
        backend="sklearn",
    )
    print("Built oxe_sim. results load:", dataset.load_brain_results("oxe_sim") is not None)

# Drive similarity from code (App panel can't resolve a frames-view index in 1.19)
res = frames_view.sort_by_similarity("laptop", k=25, brain_key="oxe_sim")
print("laptop matches:", res.count())
session.view = res

### Freeze similarity results into one-click saved views

A live `sort_by_similarity` is great for "type anything" moments, but for a scripted demo it's
safer to **freeze verified queries into saved views**: run each query once, capture the matching
IDs, and save a `select()` view of them. These reload instantly (ranked, most-similar first),
need no brain index at load time, and can't error on stage.

The dict below is a **curated list of queries confirmed to return good results** (eyeballed
once). `rubik cube` returns 25/25 actual Rubik's cubes here; `laptop` is likewise clean. Add a
new query only after eyeballing it — CLIP handles large/distinct objects well but struggles with
small objects in these cluttered robot scenes, so not every query lands.

In [ ]:
# Curated, pre-verified similarity queries -> frozen, clickable saved views
similarity_queries = {
    "laptop_search":     "laptop",
    "rubik_cube_search": "rubik cube",
}
frames_view = dataset.load_saved_view("frames_explore")
for view_name, query in similarity_queries.items():
    ids = frames_view.sort_by_similarity(query, k=25, brain_key="oxe_sim").values("id")
    dataset.save_view(
        view_name,
        dataset.load_saved_view("frames_explore").select(ids, ordered=True),
        overwrite=True,
    )
    print(f"saved '{view_name}'  ({len(ids)} frames, query={query!r})")

print("\nAll saved views:", dataset.list_saved_views())

**Distinct from the `task` field:** the per-task saved views (`task_*` from cell 7c)
filter on the dataset's own language annotations — exact, no embeddings. CLIP text search above
is fuzzy semantic image search. Two different "search by language" stories; don't conflate them.

To demo other queries live, just change the string:
`frames_view.sort_by_similarity("red cup", k=25, brain_key="oxe_sim")` then
`session.view = res`.

## 10. Scale up & reuse

- Raise `max_samples` in cell 6 (or remove it) to import all 415 episodes.
- **Reuse the pipeline for other OXE datasets**: `validate_hub_dataset()` a candidate first;
  if ✅, download → `verify_local_download()` → convert with `--push-to-hub 0 --root ...` →
  import. Clean multi-cam picks: `taco_play_lerobot` (2 cams, 3242 eps),
  `bridge_orig_lerobot` (4 cams, large).
- To compare **embodiments**, import several datasets and compute embeddings across all of
  them together — the cross-embodiment scene-diversity plot is the strongest single visual.

### Cleanup

```python
# fo.delete_dataset("oxe_fanuc_demo")   # drop the FiftyOne dataset
# session.close()                       # stop the App
```
